# 01 — Data Exploration
## Urban Resilience Engine — Nairobi

This notebook walks through the raw data sources before running the full ETL pipeline.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import sys
sys.path.insert(0, '..')
from config import *

print('Nairobi bbox:', NAIROBI_BBOX)
print('H3 resolution:', H3_RESOLUTION)

## 1. OpenStreetMap Data
Fetch building footprints, hospitals, roads, and drainage for Nairobi.

In [ ]:
# Run the OSM fetch (takes 2-5 minutes)
from src.phase1_etl.fetch_osm import fetch_buildings, fetch_hospitals, fetch_roads, fetch_drainage

buildings = fetch_buildings()
hospitals = fetch_hospitals()
roads = fetch_roads()
drainage = fetch_drainage()

print(f'Buildings: {len(buildings)}, Hospitals: {len(hospitals)}, Road segments: {len(roads)}, Drainage: {len(drainage)}')

In [ ]:
# Quick plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
buildings.plot(ax=axes[0], color='gray', alpha=0.3, markersize=0.1)
axes[0].set_title(f'Buildings ({len(buildings)})')

hospitals.plot(ax=axes[1], color='red', markersize=5)
axes[1].set_title(f'Health Facilities ({len(hospitals)})')

roads.plot(ax=axes[2], color='steelblue', linewidth=0.3)
axes[2].set_title(f'Road Network ({len(roads)} segments)')

plt.tight_layout()
plt.show()

## 2. Weather Data

In [ ]:
from src.phase1_etl.fetch_weather import fetch_daily_weather, derive_anomalies

weather = fetch_daily_weather()
weather = derive_anomalies(weather)
weather.head()

In [ ]:
fig = px.line(weather.reset_index(), x='date', y='prcp', title='Daily Precipitation — Nairobi')
fig.show()

## 3. Build H3 Grid & Merge

In [ ]:
from src.phase1_etl.clean_merge import build_grid_dataset

grid = build_grid_dataset()
grid.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
grid.plot(column='building_count', cmap='YlOrRd', legend=True, ax=ax)
ax.set_title('Building Count per H3 Hex — Nairobi')
plt.show()